In [3]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.github.ai/inference"
model='gpt-4o-mini'

openai_client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model=model,
    messages=messages
)

response.choices[0].message.content

"To determine whether you can join a course, you'll need to check the specific enrollment guidelines for that course. Typically, this information can be found on the course's official website or through the institution offering it. Look for details concerning enrollment deadlines, prerequisites, and how to register. If you're unsure, consider reaching out to the course coordinator or institution for more information."

In [34]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [35]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [30]:
# Formato correcto para .chat.completions.create
search_tool = {
    "type": "function",
    "function": { 
        "name": "search_tool",
        "description": "Busca información en la web o base de conocimiento para el RAG.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "El término o frase de búsqueda.",
                }
            },
            "required": ["query"],
        },
    },
}

In [11]:
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
)

output_message = response.choices[0].message

In [12]:
output_message

ChatCompletionMessage(content='To provide you with accurate information, could you please specify which course you are referring to?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

In [81]:
messages = [
    {
        "role": "user", 
        "content": "I am looking at the LLM Zoomcamp. Can I still join the 2026 cohort?"
    }
]

In [82]:
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
)

response.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_itiPjq73CYmH5YKNfWJjZZZW', function=Function(arguments='{"query":"LLM Zoomcamp 2026 cohort registration"}', name='search_tool'), type='function')])

In [88]:
response.choices[0].message.tool_calls[0]

ChatCompletionMessageFunctionToolCall(id='call_itiPjq73CYmH5YKNfWJjZZZW', function=Function(arguments='{"query":"LLM Zoomcamp 2026 cohort registration"}', name='search_tool'), type='function')

In [48]:
import json

call = response.choices[0].message.tool_calls[0]
args = json.loads(call.function.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [64]:
messages.append(response.choices[0].message)

In [66]:
messages.append({
    "role": "tool",            # Rol obligatorio para herramientas
    "tool_call_id": call.id,   # ID que vincula este resultado con la llamada de la IA
    "content": result_json     # El contexto real recuperado de las FAQs
})

In [68]:
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
)

output_message = response.choices[0].message

In [69]:
output_message

ChatCompletionMessage(content="Yes, you can still join the LLM Zoomcamp 2026 cohort. You don't need to formally register to start learning and submitting homework, as registration is not checked against a list. It's primarily for gauging interest. You can begin your learning and participate in the course while the form is open.\n\nFor more information about the cohort and to access resources, you can visit the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

In [73]:
response.usage

CompletionUsage(completion_tokens=106, prompt_tokens=861, total_tokens=967, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0), latency_checkpoint={'engine_tbt_ms': 12, 'engine_ttft_ms': 141, 'engine_ttlt_ms': 1407, 'pre_inference_ms': 125, 'service_tbt_ms': 12, 'service_ttft_ms': 1107, 'service_ttlt_ms': 2369, 'total_duration_ms': 2254, 'user_visible_ttft_ms': 981})

In [74]:
response.usage.prompt_tokens #input
response.usage.completion_tokens #output

106

In [75]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(response.usage.prompt_tokens, 
                                   response.usage.completion_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00019275


# Agent loop

In [76]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


In [108]:
def make_call(call):
    args = json.loads(call.function.arguments)

    if call.function.name == "search_tool":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "role": "tool",
        "tool_call_id": call.id, 
        "content": result_json
    }

In [109]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
)

assistant_message = response.choices[0].message

messages.append(assistant_message)

has_function_calls = False

if assistant_message.tool_calls:

    has_function_calls = True

    for tool_call in assistant_message.tool_calls:
        # Extraemos el nombre y los argumentos (recuerda que arguments viene como string JSON)
        func_name = tool_call.function.name
        func_args = tool_call.function.arguments
        
        print("function_call:", func_name, func_args)
               
        # Ejecutas tu función pasando el objeto 'tool_call' entero para mantener el ID
        call_output = make_call(tool_call) 
        messages.append(call_output)

# 2. EQUIVALENTE AL ELIF: Validar si el modelo simplemente respondió con texto plano
elif assistant_message.content:
    print("ASSISTANT:")
    print(assistant_message.content)

function_call: search_tool {"query":"course enrollment information"}


In [112]:
it = 1

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        tools=[search_tool],
    )
    
    assistant_message = response.choices[0].message
    
    messages.append(assistant_message)
    
    if assistant_message.tool_calls:
    
        has_function_calls = True
    
        for tool_call in assistant_message.tool_calls:
            # Extraemos el nombre y los argumentos (recuerda que arguments viene como string JSON)
            func_name = tool_call.function.name
            func_args = tool_call.function.arguments
            
            print("function_call:", func_name, func_args)
                   
            # Ejecutas tu función pasando el objeto 'tool_call' entero para mantener el ID
            call_output = make_call(tool_call) 
            messages.append(call_output)
    
    # 2. EQUIVALENTE AL ELIF: Validar si el modelo simplemente respondió con texto plano
    elif assistant_message.content:
        print("ASSISTANT:")
        last_answer = assistant_message.content
        print(assistant_message.content)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search_tool {"query":"Can I join a course after it has started?"}
iteration #2...
ASSISTANT:
Yes, you can still join the course even if you've just discovered it! However, if you wish to receive a certificate, you'll need to submit your project while submissions are still being accepted. 

Keep in mind that you can start learning and submitting homework without needing to register, as the registration is primarily to gauge interest before the course starts. 

Is there anything else you would like to know about the course?


In [115]:
def agent_loop(instructions, question, model='gpt-4o-mini') -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    it = 1
    
    while True:
        print(f"iteration #{it}...")
        has_function_calls = False
    
        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool],
        )
        
        assistant_message = response.choices[0].message
        
        messages.append(assistant_message)
        
        if assistant_message.tool_calls:
        
            has_function_calls = True
        
            for tool_call in assistant_message.tool_calls:
                # Extraemos el nombre y los argumentos (recuerda que arguments viene como string JSON)
                func_name = tool_call.function.name
                func_args = tool_call.function.arguments
                
                print("function_call:", func_name, func_args)
                       
                # Ejecutas tu función pasando el objeto 'tool_call' entero para mantener el ID
                call_output = make_call(tool_call) 
                messages.append(call_output)
        
        # 2. EQUIVALENTE AL ELIF: Validar si el modelo simplemente respondió con texto plano
        elif assistant_message.content:
            print("ASSISTANT:")
            last_answer = assistant_message.content
            print(assistant_message.content)
    
        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [116]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search_tool {"query":"run Olama locally"}
iteration #2...
ASSISTANT:
To run Olama locally, you can follow these steps:

1. **Installation**: Start by installing Olama based on your operating system:
   - **macOS**: Download the `.pkg` from [Ollama's official site](https://ollama.com/download) and install it.
   - **Windows**: Download the `.msi` and install it.
   - **Linux**: Open your terminal and run the following command:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Running the Model**: After installation, you can open a terminal and start the LLaMA 3 model by running:
   ```bash
   ollama run llama3
   ```
   This command will download the model (approximately 4GB) and start it locally.

3. **Testing the Local Server**: To verify that Ollama is running, you can use:
   ```bash
   curl http://localhost:11434
   ```
   You should receive a response that includes details about the available models.

4. **Python Client S

'To run Olama locally, you can follow these steps:\n\n1. **Installation**: Start by installing Olama based on your operating system:\n   - **macOS**: Download the `.pkg` from [Ollama\'s official site](https://ollama.com/download) and install it.\n   - **Windows**: Download the `.msi` and install it.\n   - **Linux**: Open your terminal and run the following command:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Running the Model**: After installation, you can open a terminal and start the LLaMA 3 model by running:\n   ```bash\n   ollama run llama3\n   ```\n   This command will download the model (approximately 4GB) and start it locally.\n\n3. **Testing the Local Server**: To verify that Ollama is running, you can use:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should receive a response that includes details about the available models.\n\n4. **Python Client Setup**: If you want to use Ollama with Python, install the Ollama Python cli

In [117]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search_tool {"query":"can I still join a course after it has started"}
iteration #2...
ASSISTANT:
Yes, you can still join the course even if you've just discovered it. However, if you're aiming to receive a certificate, you must submit your project while submissions are still being accepted. 

Additionally, you do not need to formally register to start learning and submitting homework, as you are accepted into the course automatically. Registration is only used to gauge interest.

Is there anything else you would like to explore regarding the course?


"Yes, you can still join the course even if you've just discovered it. However, if you're aiming to receive a certificate, you must submit your project while submissions are still being accepted. \n\nAdditionally, you do not need to formally register to start learning and submitting homework, as you are accepted into the course automatically. Registration is only used to gauge interest.\n\nIs there anything else you would like to explore regarding the course?"

In [118]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search_tool {"query":"course enrollment policies"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! However, if you want to receive a certificate, it's important that you submit your project while submissions are still being accepted. 

You don't specifically need to register to start learning; you can begin right away and submit your homework as long as the submission form is open. Registration is mainly for gauging interest before the start date. 

If you have further questions or need assistance with anything else related to the course, feel free to ask!


"Yes, you can still join the course! However, if you want to receive a certificate, it's important that you submit your project while submissions are still being accepted. \n\nYou don't specifically need to register to start learning; you can begin right away and submit your homework as long as the submission form is open. Registration is mainly for gauging interest before the start date. \n\nIf you have further questions or need assistance with anything else related to the course, feel free to ask!"

In [119]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search_tool {"query":"Queen Gambit course logistics"}
iteration #2...
function_call: search_tool {"query":"Queen Gambit chess strategy"}
iteration #3...
ASSISTANT:
It seems that my searches did not yield specific information about "Queen Gambit" in the context you are asking about. The term "Queen Gambit" is most commonly known in the context of chess as an opening strategy, but it does not seem to relate to any logistics or information about a course.

If your question is specifically about chess strategies, it falls outside the course-related content I can assist with. Please clarify your question or let me know if there are other aspects of the course or its logistics that you would like to explore!


'It seems that my searches did not yield specific information about "Queen Gambit" in the context you are asking about. The term "Queen Gambit" is most commonly known in the context of chess as an opening strategy, but it does not seem to relate to any logistics or information about a course.\n\nIf your question is specifically about chess strategies, it falls outside the course-related content I can assist with. Please clarify your question or let me know if there are other aspects of the course or its logistics that you would like to explore!'